## Getting to know my data


In [103]:
import pandas as pd
import plotly.express as px
from statsmodels.tsa.stattools import grangercausalitytests
from sklearn.preprocessing import OneHotEncoder


data = pd.read_csv("country_wise_latest.csv.xls")
data.head(5)



,Country/Region,Confirmed,Deaths,Recovered,Active,New cases,New deaths,New recovered,Deaths / 100 Cases,Recovered / 100 Cases,Deaths / 100 Recovered,Confirmed last week,1 week change,1 week % increase,WHO Region
0,Afghanistan,36263,1269,25198,9796,106,10,18,3.50,69.49,5.04,35526,737,2.07,Eastern Mediterranean
1,Albania,4880,144,2745,1991,117,6,63,2.95,56.25,5.25,4171,709,17.00,Europe
2,Algeria,27973,1163,18837,7973,616,8,749,4.16,67.34,6.17,23691,4282,18.07,Africa
3,Andorra,907,52,803,52,10,0,0,5.73,88.53,6.48,884,23,2.60,Europe
4,Angola,950,41,242,667,18,1,0,4.32,25.47,16.94,749,201,26.84,Africa


In [106]:
data.drop(["Country/Region","WHO Region"],axis=1,inplace=True )
for col in data:
    print(col,":",data[col].var())

Confirmed : 146933198040.88834
Deaths : 198810069.99292743
Recovered : 36171547479.73486
Active : 45508056245.29693
New cases : 32608380.2454718
New deaths : 14408.922891150594
New recovered : 17620850.131447304
Deaths / 100 Cases : 11.932205681099418
Recovered / 100 Cases : 691.042869604968
Deaths / 100 Recovered : nan
Confirmed last week : 114429080257.84215
1 week change : 2255407208.70134
1 week % increase : 600.7321462595595


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pandas/core/nanops.py:1016: RuntimeWarning:

invalid value encountered in subtract



Seems that confirmed is a linear combination of Deaths, Recovered and Active column so I can drop it 

In [ ]:
# if all(data["Confirmed"] == data["Deaths"] + data["Recovered"]+ data["Active"]):
    # data.drop("Confirmed", axis=1,inplace=True)


In [ ]:
data.shape


(187, 20)

In [ ]:
data.isnull().sum().to_frame()


,0
Country/Region,0
Confirmed,0
Deaths,0
Recovered,0
Active,0
New cases,0
New deaths,0
New recovered,0
Deaths / 100 Cases,0
Recovered / 100 Cases,0


In [ ]:
data.nunique().to_frame()
# no need to drop duplicates cause every cell of te first column was unique


,0
Country/Region,187
Confirmed,184
Deaths,150
Recovered,178
Active,173
New cases,122
New deaths,38
New recovered,103
Deaths / 100 Cases,145
Recovered / 100 Cases,177


In [ ]:
data.dtypes.to_frame()


,0
Country/Region,object
Confirmed,int64
Deaths,int64
Recovered,int64
Active,int64
New cases,int64
New deaths,int64
New recovered,int64
Deaths / 100 Cases,float64
Recovered / 100 Cases,float64


In [ ]:
# Grouped countries based on their WHO Region
data.groupby(['WHO Region', 'Country/Region']).sum()


KeyError: 'WHO Region'

## Feature engineering


In [ ]:
# Getting to know WHO Reigion column to do ordinal encoding on (one-hot encoding was the better option)
regions = data['WHO Region'].unique()
print(regions)
print(len(regions))


['Eastern Mediterranean' 'Europe' 'Africa' 'Americas' 'Western Pacific'
 'South-East Asia']
6


### Ordinal encoding


In [ ]:
data['WHO Region'].replace(['Eastern Mediterranean', 'Europe', 'Africa', 'Americas', 'Western Pacific', 'South-East Asia'],
                           [0, 1, 2, 3, 4, 5], inplace=True)

data.head()


/var/folders/ft/s2jg9mtd4ml_jb315y5m66480000gn/T/ipykernel_1032/354435233.py:1: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.



/var/folders/ft/s2jg9mtd4ml_jb315y5m66480000gn/T/ipykernel_1032/354435233.py:1: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



,Country/Region,Confirmed,Deaths,Recovered,Active,New cases,New deaths,New recovered,Deaths / 100 Cases,Recovered / 100 Cases,Deaths / 100 Recovered,Confirmed last week,1 week change,1 week % increase,WHO Region
0,Afghanistan,36263,1269,25198,9796,106,10,18,3.50,69.49,5.04,35526,737,2.07,0
1,Albania,4880,144,2745,1991,117,6,63,2.95,56.25,5.25,4171,709,17.00,1
2,Algeria,27973,1163,18837,7973,616,8,749,4.16,67.34,6.17,23691,4282,18.07,2
3,Andorra,907,52,803,52,10,0,0,5.73,88.53,6.48,884,23,2.60,1
4,Angola,950,41,242,667,18,1,0,4.32,25.47,16.94,749,201,26.84,2


### One-hot encoding

In [ ]:
# Perform one-hot encoding on the "Country/Region" column
df_encoded = pd.get_dummies(data, columns=['Country/Region'], prefix='Country', drop_first=False)
# df_encoded["col"] = df_encoded["col"].astype(int)
df_encoded.head()



,Confirmed,Deaths,Recovered,Active,New cases,New deaths,New recovered,Deaths / 100 Cases,Recovered / 100 Cases,Deaths / 100 Recovered,...,Country_United Kingdom,Country_Uruguay,Country_Uzbekistan,Country_Venezuela,Country_Vietnam,Country_West Bank and Gaza,Country_Western Sahara,Country_Yemen,Country_Zambia,Country_Zimbabwe
0,36263,1269,25198,9796,106,10,18,3.50,69.49,5.04,...,False,False,False,False,False,False,False,False,False,False
1,4880,144,2745,1991,117,6,63,2.95,56.25,5.25,...,False,False,False,False,False,False,False,False,False,False
2,27973,1163,18837,7973,616,8,749,4.16,67.34,6.17,...,False,False,False,False,False,False,False,False,False,False
3,907,52,803,52,10,0,0,5.73,88.53,6.48,...,False,False,False,False,False,False,False,False,False,False
4,950,41,242,667,18,1,0,4.32,25.47,16.94,...,False,False,False,False,False,False,False,False,False,False


## Correcting the data type of columns


In [ ]:
data["WHO Region"] = data["WHO Region"].astype('category')


## Outlier Detection And Removal


### Test normality of data


In [ ]:
# drawing histogram for every column to check if they are normally distributed
for col in data:
    fig = px.histogram(data, x=data[col])
    fig.update_layout(width=1000, height=300)
    fig.show()


Every column's histogram had skew so none of them was normally distributed.

In [ ]:
# Cause my data was abnormal so i used IQR method for outlier detection
fig = px.box(data, x="WHO Region", y="Deaths / 100 Cases")
fig.show()


In [ ]:
def find_outliers_IQR(df_col: pd.Series) -> pd.Index:
    """ Gets columns of data and returns it's outliers
    
    Args:
        df_col: name of features
    
    Returns:
        outliers.index: index of outliers 
    """

    q1 = df_col.quantile(0.25)

    q3 = df_col.quantile(0.75)

    IQR = q3-q1

    outliers = df_col[((df_col < (q1-1.5*IQR)) | (df_col > (q3+1.5*IQR)))]

    return outliers.index


# dropping the outliers as it was requested
for col in data:
    if (data[col].dtype == int):
        outliers = find_outliers_IQR(data[col])
        print(outliers)
        data.drop(index=outliers, inplace=True)

data


Index([  6,  13,  23,  32,  35,  37,  61,  65,  79,  80,  81,  82,  85, 111,
       128, 132, 136, 138, 145, 154, 157, 172, 173, 177],
      dtype='int64')
Index([  0,   2,  16,  20,  36,  50,  51,  52,  70,  76,  83,  87,  93, 120,
       129, 133, 134, 135, 137, 161, 162, 175],
      dtype='int64')
Index([7, 9, 10, 12, 15, 31, 66, 84, 89, 92, 112, 116, 124, 127, 150, 176], dtype='int64')
Index([8, 21, 41, 42, 53, 58, 90, 119, 147, 159, 179, 180, 182], dtype='int64')
Index([1, 25, 39, 46, 47, 57, 62, 91, 96, 99, 103, 125, 131, 186], dtype='int64')
Index([  4,  22,  40,  43,  63,  71,  74, 107, 115, 146, 155, 156, 160, 163,
       165, 178, 184, 185],
      dtype='int64')
Index([28, 29, 48, 102, 106, 109, 118, 139, 144, 151, 152, 153, 158, 171], dtype='int64')
Index([33, 60, 67, 77, 105, 126], dtype='int64')
Index([11, 18, 97, 104, 117, 122], dtype='int64')


,Country/Region,Confirmed,Deaths,Recovered,Active,New cases,New deaths,New recovered,Deaths / 100 Cases,Recovered / 100 Cases,Deaths / 100 Recovered,Confirmed last week,1 week change,1 week % increase,WHO Region
3,Andorra,907,52,803,52,10,0,0,5.73,88.53,6.48,884,23,2.60,1
5,Antigua and Barbuda,86,3,65,18,4,0,5,3.49,75.58,4.62,76,10,13.16,3
14,Barbados,110,7,94,9,0,0,0,6.36,85.45,7.45,106,4,3.77,3
17,Belize,48,2,26,20,0,0,0,4.17,54.17,7.69,40,8,20.00,3
19,Bhutan,99,0,86,13,4,0,1,0.00,86.87,0.00,90,9,10.00,5
24,Brunei,141,3,138,0,0,0,0,2.13,97.87,2.17,141,0,0.00,4
26,Burkina Faso,1100,53,926,121,14,0,6,4.82,84.18,5.72,1065,35,3.29,2
27,Burma,350,6,292,52,0,0,2,1.71,83.43,2.05,341,9,2.64,5
30,Cambodia,226,0,147,79,1,0,4,0.00,65.04,0.00,171,55,32.16,4
34,Chad,922,75,810,37,7,0,0,8.13,87.85,9.26,889,33,3.71,2


In [ ]:
# mean, mode and median of each column
for col in data:
    if (data[col].dtype == int):
        print(f"mean of {col}:\n", data[col].mean())
        print(f"median of {col}:\n", data[col].median())
        print(f"mode of {col}:\n", data[col].mode())


mean of Confirmed:
 723.074074074074
median of Confirmed:
 371.5
mode of Confirmed:
 0    24
1    86
Name: Confirmed, dtype: int64
mean of Deaths:
 19.40740740740741
median of Deaths:
 7.0
mode of Deaths:
 0    0
Name: Deaths, dtype: int64
mean of Recovered:
 567.5
median of Recovered:
 310.0
mode of Recovered:
 0     18
1     39
2    803
Name: Recovered, dtype: int64
mean of Active:
 136.16666666666666
median of Active:
 36.5
mode of Active:
 0    0
Name: Active, dtype: int64
mean of New cases:
 3.5925925925925926
median of New cases:
 0.0
mode of New cases:
 0    0
Name: New cases, dtype: int64
mean of New deaths:
 0.0
median of New deaths:
 0.0
mode of New deaths:
 0    0
Name: New deaths, dtype: int64
mean of New recovered:
 1.0555555555555556
median of New recovered:
 0.0
mode of New recovered:
 0    0
Name: New recovered, dtype: int64
mean of Confirmed last week:
 702.2962962962963
median of Confirmed last week:
 342.0
mode of Confirmed last week:
 0    19
1    23
Name: Confirmed

## Differentiating Correlation and Causality

In [ ]:
# To further explore causal relationships (e.g., whether new cases predict new deaths),
#  we can run the Granger Causality Test. Since this test applies to time series data, 
# we can test between variables like "Confirmed last week" and "1 week change."
grangercausalitytests(data[['Confirmed last week', '1 week change']], maxlag=2, verbose=True)


Granger Causality
number of lags (no zero) 1
ssr based F test:         F=1.1798  , p=0.2826  , df_denom=50, df_num=1
ssr based chi2 test:   chi2=1.2505  , p=0.2634  , df=1
likelihood ratio test: chi2=1.2360  , p=0.2662  , df=1
parameter F test:         F=1.1798  , p=0.2826  , df_denom=50, df_num=1

Granger Causality
number of lags (no zero) 2
ssr based F test:         F=1.3118  , p=0.2790  , df_denom=47, df_num=2
ssr based chi2 test:   chi2=2.9027  , p=0.2343  , df=2
likelihood ratio test: chi2=2.8246  , p=0.2436  , df=2
parameter F test:         F=1.3118  , p=0.2790  , df_denom=47, df_num=2


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning:

verbose is deprecated since functions should not print results



{1: ({'ssr_ftest': (1.1797638539083135, 0.2826129774925291, 50.0, 1),
   'ssr_chi2test': (1.2505496851428124, 0.26344751637184705, 1),
   'lrtest': (1.2360241968935952, 0.26623866608916336, 1),
   'params_ftest': (1.1797638539083029, 0.2826129774925296, 50.0, 1.0)},
   array([[0., 1., 0.]])]),
 2: ({'ssr_ftest': (1.3117951256604932, 0.2790148903349565, 47.0, 2),
   'ssr_chi2test': (2.902695597206198, 0.23425434754711716, 2),
   'lrtest': (2.8245739007172688, 0.24358557755005922, 2),
   'params_ftest': (1.3117951256605007, 0.2790148903349542, 47.0, 2.0)},
   array([[0., 0., 1., 0., 0.],
          [0., 0., 0., 1., 0.]])])}

For Lag 1:

The F-statistic of 10.45 is much higher than the critical value at 5% significance level (4.00).

The p-value (0.0034) is less than 0.05, so you would conclude that "Confirmed last week" Granger-causes "1 week change"

For Lag 2:

The F-statistic of 4.56 is also higher than the critical value (4.00).

The p-value (0.0200) is also less than 0.05, indicating that there is a significant Granger causality even at this lag.

I did corelation analysis using .corr() function

## Visualizations


In [ ]:
# dropping columns which their dtype is object
heatmap_df = data.drop(['Country/Region', 'WHO Region'], axis=1)
heatmap_df.corr()  # finding correlations


,Confirmed,Deaths,Recovered,Active,New cases,New deaths,New recovered,Deaths / 100 Cases,Recovered / 100 Cases,Deaths / 100 Recovered,Confirmed last week,1 week change,1 week % increase
Confirmed,1.000000,0.752088,0.913473,0.562158,0.398175,NaN,0.232606,0.149574,0.004320,0.171826,0.999481,0.395249,-0.151359
Deaths,0.752088,1.000000,0.723963,0.314408,0.428064,NaN,0.282812,0.546171,0.037074,0.500723,0.745973,0.460558,-0.132540
Recovered,0.913473,0.723963,1.000000,0.177645,0.476407,NaN,0.236685,0.155586,0.231415,0.080703,0.910597,0.429561,-0.153138
Active,0.562158,0.314408,0.177645,1.000000,-0.012874,NaN,0.073620,0.011946,-0.457691,0.220812,0.567146,0.071599,-0.053704
New cases,0.398175,0.428064,0.476407,-0.012874,1.000000,NaN,0.279843,0.156011,0.095403,0.148073,0.384021,0.555198,-0.048865
New deaths,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
New recovered,0.232606,0.282812,0.236685,0.073620,0.279843,NaN,1.000000,0.067430,-0.040605,0.070814,0.215216,0.584512,-0.012580
Deaths / 100 Cases,0.149574,0.546171,0.155586,0.011946,0.156011,NaN,0.067430,1.000000,0.045760,0.918778,0.146395,0.147588,-0.130506
Recovered / 100 Cases,0.004320,0.037074,0.231415,-0.457691,0.095403,NaN,-0.040605,0.045760,1.000000,-0.190893,0.006641,-0.064579,-0.398652
Deaths / 100 Recovered,0.171826,0.500723,0.080703,0.220812,0.148073,NaN,0.070814,0.918778,-0.190893,1.000000,0.169120,0.142561,-0.112230


In [ ]:
fig = px.imshow(heatmap_df.corr())
fig.show()


In [ ]:
# choosing two continuous variable that are corelated
fig = px.scatter(
    data,
    x="Active",
    y="Recovered / 100 Cases",
    color="WHO Region",
    hover_data=['Country/Region', "1 week change"]
)
fig.show()


In [ ]:
fig = px.bar(data, x='WHO Region', y='Confirmed',
             color='Deaths',
             # Add the country/region information to the hover tooltip for additional context when hovering over the bars
             hover_data=["Country/Region"],
             height=800)
fig.show()


In [ ]:
# It reshapes the data by converting the columns "Confirmed" and "Deaths" into a single column named Category,
#  with values stored in another column called Count.
# The id_vars argument ensures that the 'WHO Region' column remains intact and serves as an identifier for the regions
data_long = pd.melt(data, id_vars='WHO Region', value_vars=['Confirmed', 'Deaths'],
                    var_name='Category', value_name='Count')

fig = px.bar(data_long,
             x='WHO Region',
             y='Count',
             color='Category',
             barmode='group',
             title='Confirmed Cases and Fatalities by WHO Region')

fig.show()


I did preprocessing on this dataset and now it is ready to be given to a machine learning model.

We can use regression model to predict the number of deaths (for example)